# Final Integration Inference
Upload one supported file and this notebook will choose and execute the matching model notebook.

Supported routing: `.log -> BNN`, image -> CNN, `.csv -> SVM`, `.wav -> MLP`.

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

integration_dir = Path.cwd().resolve()
if str(integration_dir) not in sys.path:
    sys.path.insert(0, str(integration_dir))

import integration_runtime as ir

supported_routes = {
    ".log": "BNN CAN notebook",
    "image": "CNN traffic-sign notebook",
    ".csv": "SVM lane notebook",
    ".wav": "MLP siren notebook",
}

print("Supported routing:")
for key, value in supported_routes.items():
    print(f"  {key:<6} -> {value}")

In [ ]:
upload = widgets.FileUpload(
    accept=".log,.csv,.wav,.png,.jpg,.jpeg,.bmp,.gif,.webp",
    multiple=False,
    description="Upload File",
)
run_button = widgets.Button(description="Run Routed Notebook", button_style="primary")
output = widgets.Output()


def _uploaded_entry(file_upload):
    value = file_upload.value
    if isinstance(value, dict):
        if not value:
            return None
        return next(iter(value.values()))
    if isinstance(value, tuple):
        if not value:
            return None
        return value[0]
    return None


def _clear_upload(file_upload):
    try:
        file_upload.value.clear()
    except Exception:
        pass
    try:
        file_upload._counter = 0
    except Exception:
        pass


def _run_selected_notebook(_):
    with output:
        output.clear_output()
        uploaded = _uploaded_entry(upload)
        if uploaded is None:
            print("Upload one supported file first.")
            return

        filename = uploaded.get("name", "uploaded_input")
        content = uploaded.get("content", b"")

        try:
            saved_path = ir.save_uploaded_file(filename, content)
            model_key = ir.infer_model_from_filename(filename)
            notebook_path = Path(ir.notebook_links()[model_key])

            print(f"Saved upload: {saved_path}")
            print(f"Selected model: {ir.MODEL_CONFIG[model_key]['label']}")
            print(f"Notebook: {notebook_path}")
            print("Executing selected notebook...\n")

            notebook_result = ir.execute_model_notebook(model_key, saved_path)

            print("Notebook execution finished.")
            print(f"Executed notebook: {notebook_result['executed_notebook']}")

            stdout = notebook_result.get("stdout", "").strip()
            stderr = notebook_result.get("stderr", "").strip()
            if stdout:
                print("\nNotebook stdout:\n")
                print(stdout)
            if stderr:
                print("\nNotebook stderr:\n")
                print(stderr)
        except Exception as exc:
            print(f"Routing or execution failed: {exc}")
        finally:
            _clear_upload(upload)


run_button.on_click(_run_selected_notebook)

display(upload, run_button, output)